# LeetCode 1850 — Minimum Adjacent Swaps to Reach the Kth Smallest Number bằng Python

Link đề: https://leetcode.com/problems/minimum-adjacent-swaps-to-reach-the-kth-smallest-number/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def getMinSwaps(self, num: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Cài thuật toán **hoán vị kế tiếp** (next permutation) và dùng nó `k` lần để có số "wonderful" nhỏ thứ `k`.
2. Hiểu bài toán con: **số lần hoán đổi kề nhau tối thiểu** để biến chuỗi `A` thành chuỗi `B` (cùng multiset ký tự) = số **nghịch thế** (inversions) của phép ghép vị trí.
3. Nắm quy tắc ghép **tham lam**: ký tự giống nhau ghép theo thứ tự xuất hiện (lần thứ `t` của `c` trong `B` ↔ lần thứ `t` của `c` trong `A`) — không bao giờ cần "vượt" hai ký tự giống nhau.
4. Tính số lần đổi bằng mô phỏng **nổi bọt** `O(n²)`, rồi tối ưu bằng **cây Fenwick** / **merge sort** đếm nghịch thế `O(n log n)`.
5. Kiểm chứng bằng vét cạn BFS trên đồ thị hoán đổi (chuỗi ngắn) và liệt kê hoán vị (`itertools`).

## 2. Đề bài & ví dụ

Cho chuỗi `num` biểu diễn một số nguyên lớn và số nguyên `k`. Một số nguyên gọi là **wonderful** nếu nó là một **hoán vị các chữ số** của `num` và **lớn hơn** `num`. Xét các số wonderful nhỏ nhất: thứ 1, thứ 2, …

Ví dụ `num = "5489355142"`: wonderful nhỏ thứ 1 là `"5489355214"`, thứ 2 `"5489355241"`, thứ 3 `"5489355412"`, thứ 4 `"5489355421"`.

Trả về **số lần hoán đổi hai chữ số kề nhau ít nhất** để biến `num` thành số wonderful nhỏ thứ `k`. Đề đảm bảo số đó tồn tại.

**Ví dụ 1:** `num = "5489355142", k = 4` → `2` (đổi vị trí 7–8 rồi 8–9: `"5489355142" → "5489355412" → "5489355421"`).

**Ví dụ 2:** `num = "11112", k = 4` → `4` (đích `"21111"`, dời chữ 2 qua 4 vị trí).

**Ví dụ 3:** `num = "00123", k = 1` → `1` (đích `"00132"`).

**Ràng buộc:** `2 ≤ num.length ≤ 1000`; `1 ≤ k ≤ 1000`; `num` chỉ gồm chữ số.

In [1]:
import random
import time
from collections import deque, defaultdict
from itertools import permutations
from typing import List

examples = [("5489355142", 4, 2), ("11112", 4, 4), ("00123", 1, 1)]
for num, k, exp in examples:
    print(f"num={num!r}, k={k} -> kỳ vọng {exp}")

num='5489355142', k=4 -> kỳ vọng 2
num='11112', k=4 -> kỳ vọng 4
num='00123', k=1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Hoán vị kế tiếp (next permutation)

Từ dãy `a`, hoán vị lớn hơn **nhỏ nhất**:
1. Tìm `i` lớn nhất với `a[i] < a[i+1]` (nếu không có, `a` đã là hoán vị lớn nhất).
2. Tìm `j > i` lớn nhất với `a[j] > a[i]`; đổi `a[i]`, `a[j]`.
3. Đảo ngược đoạn `a[i+1:]` (đang giảm dần) thành tăng dần.

In [2]:
def next_permutation(a):
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i < 0:
        return False
    j = len(a) - 1
    while a[j] <= a[i]:
        j -= 1
    a[i], a[j] = a[j], a[i]
    a[i + 1:] = reversed(a[i + 1:])
    return True

a = list("5489355142")
for t in range(1, 5):
    next_permutation(a); print(t, "".join(a))

1 5489355214
2 5489355241
3 5489355412
4 5489355421


### 3.2 Hoán đổi kề nhau và nghịch thế

Một lần đổi hai phần tử kề nhau làm số **nghịch thế** (cặp `i < j` mà `p[i] > p[j]`) thay đổi đúng ±1. Vì vậy số lần đổi kề **tối thiểu** để sắp một hoán vị `p` về `0, 1, …, n−1` bằng **số nghịch thế** của `p` (sắp nổi bọt đạt đúng con số này).

In [3]:
def inversions_naive(p):
    return sum(1 for i in range(len(p)) for j in range(i + 1, len(p)) if p[i] > p[j])

print(inversions_naive([2, 0, 1]), inversions_naive([0, 1, 2]), inversions_naive([3, 2, 1, 0]))

2 0 6


### 3.3 Ghép vị trí khi có ký tự lặp

Biến `A` thành `B` (cùng multiset): ghép mỗi vị trí của `B` với một vị trí của `A` mang cùng ký tự. Với ký tự lặp, ghép **theo thứ tự xuất hiện** (lần thứ `t` ↔ lần thứ `t`) — vượt qua nhau hai bản sao giống hệt là phí công. Khi đó `p[i]` = vị trí trong `A` của ký tự đặt tại vị trí `i` của `B`, và đáp án = `inversions(p)`.

In [4]:
def match_positions(A, B):
    pos = defaultdict(deque)
    for i, c in enumerate(A):
        pos[c].append(i)
    return [pos[c].popleft() for c in B]

print(match_positions("11112", "21111"), "->", inversions_naive(match_positions("11112", "21111")))

[4, 0, 1, 2, 3] -> 4


### 3.4 Cây Fenwick đếm nghịch thế

Duyệt `p` từ trái sang phải; với mỗi `p[i]`, số phần tử **đã gặp** lớn hơn `p[i]` = `i − (số đã gặp ≤ p[i])`. BIT trả lời "số đã gặp ≤ x" trong `O(log n)`.

In [5]:
def inversions_bit(p):
    n = len(p); t = [0] * (n + 1); inv = 0
    for i, x in enumerate(p):
        j = x + 1; le = 0
        while j > 0: le += t[j]; j -= j & -j
        inv += i - le
        j = x + 1
        while j <= n: t[j] += 1; j += j & -j
    return inv

print(inversions_bit([2, 0, 1]), inversions_bit([3, 2, 1, 0]), inversions_bit(list(range(5))))

2 6 0


## 4. Ý tưởng nền tảng / trực giác

Bài gồm **hai bài con độc lập**:
1. Tìm đích `B` = hoán vị kế tiếp thứ `k` của `num` — chỉ cần gọi `next_permutation` `k` lần (`k ≤ 1000`, mỗi lần `O(n)`).
2. Đếm số lần đổi kề tối thiểu từ `num` sang `B`.

Tính tay **Ví dụ 2**: `num = "11112"`, đích `"21111"`. Ghép: chữ `2` của đích (vị trí 0) ← vị trí 4 của `num`; bốn chữ `1` ← vị trí 0, 1, 2, 3. `p = [4, 0, 1, 2, 3]` có 4 nghịch thế (4 > 0, 1, 2, 3) ⇒ 4 lần đổi.

**Ví dụ 1:** `num = "5489355142"`, đích `"5489355421"`. Bảy chữ đầu giống nhau; phần cuối `"142" → "421"`: `p` phần cuối = `[8, 9, 7]` ⇒ nghịch thế 2.

In [6]:
for num, k, exp in examples:
    b = list(num)
    for _ in range(k): next_permutation(b)
    B = "".join(b); p = match_positions(num, B)
    print(f"{num} -> đích {B}, p={p}, nghịch thế={inversions_naive(p)} (kỳ vọng {exp})")

5489355142 -> đích 5489355421, p=[0, 1, 2, 3, 4, 5, 6, 8, 9, 7], nghịch thế=2 (kỳ vọng 2)
11112 -> đích 21111, p=[4, 0, 1, 2, 3], nghịch thế=4 (kỳ vọng 4)
00123 -> đích 00132, p=[0, 1, 2, 4, 3], nghịch thế=1 (kỳ vọng 1)


## 5. Cách 1 — Brute Force / baseline

### 5.1 Đích bằng liệt kê mọi hoán vị; số lần đổi bằng BFS

Cho chuỗi **ngắn** (`n ≤ 7`): sinh mọi hoán vị khác nhau, sắp, lấy hoán vị thứ `k` lớn hơn `num`; rồi BFS trên đồ thị "chuỗi ↔ chuỗi sau một lần đổi kề" để có số bước ít nhất — đúng định nghĩa, không dùng lý thuyết nghịch thế.

In [7]:
def kth_wonderful_brute(num, k):
    bigger = sorted({"".join(p) for p in permutations(num) if "".join(p) > num})
    return bigger[k - 1]

def min_swaps_bfs(A, B):
    dist = {A: 0}; q = deque([A])
    while q:
        s = q.popleft()
        if s == B:
            return dist[s]
        for i in range(len(s) - 1):
            if s[i] != s[i + 1]:
                t = s[:i] + s[i + 1] + s[i] + s[i + 2:]
                if t not in dist:
                    dist[t] = dist[s] + 1; q.append(t)
    return -1

def swaps_bruteforce(num, k):
    return min_swaps_bfs(num, kth_wonderful_brute(num, k))

for num, k, exp in examples[1:]:
    got = swaps_bruteforce(num, k); print(num, k, "->", got, "| kỳ vọng", exp); assert got == exp

11112 4 -> 4 | kỳ vọng 4
00123 1 -> 1 | kỳ vọng 1


**Độ phức tạp:** liệt kê `n!` hoán vị và BFS trên tới `n!` trạng thái — chỉ dùng cho `n ≤ 8`.

### 5.2 Baseline thực dụng: `next_permutation` `k` lần + mô phỏng nổi bọt

Dùng `next_permutation` cho đích. Rồi với mỗi vị trí `i` từ trái: nếu `cur[i] != B[i]`, tìm `j > i` gần nhất có `cur[j] == B[i]`, dời nó về `i` bằng `j − i` lần đổi kề.

In [8]:
def swaps_bubble(num, k):
    b = list(num)
    for _ in range(k):
        next_permutation(b)
    cur = list(num); n = len(cur); ans = 0
    for i in range(n):
        if cur[i] == b[i]:
            continue
        j = i + 1
        while cur[j] != b[i]:
            j += 1
        while j > i:                                  # dời cur[j] về i
            cur[j], cur[j - 1] = cur[j - 1], cur[j]
            j -= 1; ans += 1
    return ans

for num, k, exp in examples:
    got = swaps_bubble(num, k); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
4 kỳ vọng 4
1 kỳ vọng 1


**Độ phức tạp:** `O(k · n + n²)` — với `n = 1000` khoảng `10⁶`, đã qua giới hạn. Đây là lời giải phổ biến nhất cho bài.

**Nút thắt (nếu `n` lớn hơn):** phần đếm đổi là `O(n²)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đích):** hoán vị kế tiếp `k` lần — `O(k · n)`.

**Insight 2 (ghép tham lam ký tự lặp):** khi dời ký tự `c` về vị trí `i`, luôn chọn bản sao `c` **gần nhất** bên phải. Chọn bản sao xa hơn phải vượt qua bản sao gần — vô ích. Tương đương ghép lần thứ `t` của `c` trong đích với lần thứ `t` của `c` trong `num`.

**Insight 3 (đếm = nghịch thế):** sau khi ghép, `p[i]` = vị trí gốc của ký tự đứng ở `i` trong đích. Số lần đổi kề tối thiểu = số nghịch thế của `p` ⇒ đếm bằng BIT hoặc merge sort `O(n log n)`.

**Quy tắc:** "nếu ký tự đã đúng chỗ thì bỏ qua; nếu chưa thì kéo bản sao gần nhất về" (mô phỏng) ⇔ "đếm số phần tử vượt qua nó" (nghịch thế).

**Invariant (mô phỏng):** sau bước `i`, `cur[0..i] == B[0..i]` và thứ tự tương đối của phần còn lại giữ nguyên như ban đầu.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ghép vị trí + đếm nghịch thế bằng Fenwick

In [9]:
def swaps_bit(num, k):
    b = list(num)
    for _ in range(k):
        next_permutation(b)
    return inversions_bit(match_positions(num, b))

print([swaps_bit(n, k) for n, k, _ in examples])

[2, 4, 1]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** `O(k·n + n log n)`.

### 7.2 Cách 3 — Đếm nghịch thế bằng merge sort

In [10]:
def inversions_merge(p):
    def sort(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        L, x = sort(a[:mid]); R, y = sort(a[mid:])
        merged, i, j, inv = [], 0, 0, x + y
        while i < len(L) and j < len(R):
            if L[i] <= R[j]:
                merged.append(L[i]); i += 1
            else:
                merged.append(R[j]); j += 1; inv += len(L) - i      # mọi L[i:] > R[j]
        merged += L[i:] + R[j:]
        return merged, inv
    return sort(list(p))[1]

def swaps_merge(num, k):
    b = list(num)
    for _ in range(k): next_permutation(b)
    return inversions_merge(match_positions(num, b))

print([swaps_merge(n, k) for n, k, _ in examples])

[2, 4, 1]


**Độ phức tạp Cách 3:** `O(k·n + n log n)`; đệ quy sâu `log n` — an toàn.

### 7.3 Cách 4 — Đếm trực tiếp theo cặp (nghịch thế `O(n²)`, không mô phỏng)

Không dời ký tự, chỉ đếm `#{(i, j): i < j, p[i] > p[j]}` — cùng độ phức tạp với Cách 1b nhưng code ngắn và không sửa mảng; tiện để giải thích rằng "số lần đổi = số cặp phải vượt qua nhau".

In [11]:
def swaps_pairs(num, k):
    b = list(num)
    for _ in range(k): next_permutation(b)
    return inversions_naive(match_positions(num, b))

print([swaps_pairs(n, k) for n, k, _ in examples])

[2, 4, 1]


**Độ phức tạp Cách 4:** `O(k·n + n²)`.

## 8. TRACE từng bước

Trace phần mô phỏng nổi bọt trên **Ví dụ 3** và trên ví dụ tự dựng `num = "1232", k = 3` (ví dụ tự dựng, đối chiếu BFS; nên đối chiếu lại đề gốc):

In [12]:
def trace_bubble(num, k):
    b = list(num)
    for t in range(k):
        next_permutation(b); print(f"  next_permutation lần {t+1}: {''.join(b)}")
    cur = list(num); ans = 0
    for i in range(len(cur)):
        if cur[i] == b[i]:
            print(f"  i={i}: '{cur[i]}' đã đúng"); continue
        j = i + 1
        while cur[j] != b[i]: j += 1
        print(f"  i={i}: cần '{b[i]}', lấy bản gần nhất ở j={j} -> {j - i} lần đổi", end="")
        while j > i:
            cur[j], cur[j - 1] = cur[j - 1], cur[j]; j -= 1; ans += 1
        print(f" | cur={''.join(cur)}, tổng={ans}")
    return ans

print("Ví dụ 3:"); assert trace_bubble("00123", 1) == 1
print("Tự dựng '1232', k=3:"); r = trace_bubble("1232", 3); assert r == swaps_bruteforce("1232", 3); print("BFS xác nhận:", r)

Ví dụ 3:
  next_permutation lần 1: 00132
  i=0: '0' đã đúng
  i=1: '0' đã đúng
  i=2: '1' đã đúng
  i=3: cần '3', lấy bản gần nhất ở j=4 -> 1 lần đổi | cur=00132, tổng=1
  i=4: '2' đã đúng
Tự dựng '1232', k=3:
  next_permutation lần 1: 1322
  next_permutation lần 2: 2123
  next_permutation lần 3: 2132
  i=0: cần '2', lấy bản gần nhất ở j=1 -> 1 lần đổi | cur=2132, tổng=1
  i=1: '1' đã đúng
  i=2: '3' đã đúng
  i=3: '2' đã đúng
BFS xác nhận: 1


Bảng ghép vị trí + nghịch thế cho `"1232" → đích`:

In [13]:
b = list("1232")
for _ in range(3): next_permutation(b)
p = match_positions("1232", b)
print("đích:", "".join(b), "| p =", p, "| các cặp nghịch thế:", [(p[i], p[j]) for i in range(4) for j in range(i + 1, 4) if p[i] > p[j]])

đích: 2132 | p = [1, 0, 2, 3] | các cặp nghịch thế: [(1, 0)]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — next permutation `k` lần + ghép vị trí + Fenwick.

In [14]:
class Solution:
    def getMinSwaps(self, num: str, k: int) -> int:
        b = list(num)
        for _ in range(k):
            i = len(b) - 2
            while i >= 0 and b[i] >= b[i + 1]:
                i -= 1
            j = len(b) - 1
            while b[j] <= b[i]:
                j -= 1
            b[i], b[j] = b[j], b[i]
            b[i + 1:] = reversed(b[i + 1:])
        pos = defaultdict(deque)
        for idx, c in enumerate(num):
            pos[c].append(idx)
        p = [pos[c].popleft() for c in b]
        n = len(p)
        tree = [0] * (n + 1)
        inv = 0
        for i, x in enumerate(p):
            j = x + 1
            le = 0
            while j > 0:
                le += tree[j]
                j -= j & -j
            inv += i - le
            j = x + 1
            while j <= n:
                tree[j] += 1
                j += j & -j
        return inv

for num, k, exp in examples:
    got = Solution().getMinSwaps(num, k); print(got, exp); assert got == exp

2 2
4 4
1 1


## 10. Vì sao thuật toán đúng

**Phần 1 (đích).** `next_permutation` trả hoán vị **lớn hơn nhỏ nhất** (chứng minh chuẩn: phần hậu tố từ `i+1` đang giảm dần là hoán vị lớn nhất của nó; tăng `a[i]` lên ít nhất có thể rồi đặt hậu tố nhỏ nhất). Áp dụng `k` lần cho wonderful nhỏ thứ `k` (đề đảm bảo tồn tại nên không bao giờ "hết" hoán vị).

**Phần 2 (số lần đổi).** Với hoán vị `p` các **chỉ số** (mọi phần tử khác nhau), mỗi lần đổi kề thay đổi số nghịch thế đúng ±1, và nổi bọt giảm đúng 1 mỗi bước ⇒ min = `inv(p)`.

Với ký tự lặp, cần chọn cách ghép tốt nhất. **Bổ đề hoán đổi:** trong một ghép, nếu hai bản sao của cùng ký tự `c` "bắt chéo" (bản đứng trước trong đích ghép với bản đứng sau trong `num`), đổi hai ghép cho nhau không làm tăng số nghịch thế (cặp bắt chéo đó tự nó là một nghịch thế; mọi phần tử khác so với hai vị trí này: số nghịch thế không tăng). Vậy ghép theo thứ tự xuất hiện (không bắt chéo) là tối ưu ⇒ đáp án = `inv(p)` với `p` của ghép này. Bộ test đối chiếu với BFS xác nhận thêm. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(num) ≤ 1000`, `k ≤ 1000`.

| Cách | Time | Space |
|---|---|---|
| 1a. Liệt kê + BFS | `O(n! · n)` | `O(n!)` |
| 1b. Next perm + nổi bọt | `O(k·n + n²)` | `O(n)` |
| 2. Ghép + Fenwick | `O(k·n + n log n)` | `O(n)` |
| 3. Ghép + merge sort | `O(k·n + n log n)` | `O(n)` |
| 4. Ghép + đếm cặp | `O(k·n + n²)` | `O(n)` |

Phần `k·n` (tối đa `10⁶`) thường lớn hơn phần đếm; trong thực tế `next_permutation` chỉ chạm hậu tố ngắn nên nhanh hơn nhiều.

In [15]:
rng = random.Random(1850)
big = "".join(rng.choice("0123456789") for _ in range(1000))
rev = "".join(sorted(big[:900], reverse=False)) + "".join(sorted(big[900:], reverse=True))
for name, f in [("bubble", swaps_bubble), ("BIT", Solution().getMinSwaps), ("merge", swaps_merge), ("pairs", swaps_pairs)]:
    for label, s in [("ngẫu nhiên", big), ("có cấu trúc", rev)]:
        t = time.perf_counter(); r = f(s, 1000); print(f"{name:<6} {label:<11}: {r:>7} ({(time.perf_counter()-t)*1000:.0f} ms)")

bubble ngẫu nhiên :       9 (0 ms)
bubble có cấu trúc:   12425 (4 ms)
BIT    ngẫu nhiên :       9 (1 ms)
BIT    có cấu trúc:   12425 (3 ms)
merge  ngẫu nhiên :       9 (1 ms)
merge  có cấu trúc:   12425 (3 ms)
pairs  ngẫu nhiên :       9 (11 ms)
pairs  có cấu trúc:   12425 (13 ms)


## 12. Bộ test

Đối chiếu với vét cạn (liệt kê + BFS) trên chuỗi ngắn (`n ≤ 6`, `k` hợp lệ), và giữa các cách nhanh trên chuỗi dài. Hàm sinh chỉ tạo test **hợp lệ**: `k` không vượt số wonderful hiện có.

In [16]:
ALL = {"bubble": swaps_bubble, "bit": swaps_bit, "merge": swaps_merge, "pairs": swaps_pairs, "submit": Solution().getMinSwaps}
def run_all(num, k, brute=True):
    res = {name: f(num, k) for name, f in ALL.items()}
    if brute:
        res["brute"] = swaps_bruteforce(num, k)
    assert len(set(res.values())) == 1, (num, k, res)
    return res["submit"]

for num, k, exp in examples:
    assert run_all(num, k, brute=len(num) <= 6) == exp
edge = [("12", 1, 1), ("1111112", 1, 1), ("123", 5, 3), ("0011", 5, 4), ("2112", 1, None)]
for num, k, exp in edge:
    cnt = len({"".join(p) for p in permutations(num) if "".join(p) > num})
    if k <= cnt:
        r = run_all(num, k)
        if exp is not None: assert r == exp, (num, k, r)
rng = random.Random(18500)
done = 0
while done < 250:
    n = rng.randint(2, 6); num = "".join(rng.choice("0123" if rng.random() < 0.5 else "0123456789") for _ in range(n))
    cnt = len({"".join(p) for p in permutations(num) if "".join(p) > num})
    if cnt == 0: continue
    run_all(num, rng.randint(1, min(cnt, 30))); done += 1
for _ in range(10):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(50, 400)))
    run_all(s, rng.randint(1, 200), brute=False)
print("Ví dụ, edge case, 250 test vét cạn BFS, 10 test dài: OK")

Ví dụ, edge case, 250 test vét cạn BFS, 10 test dài: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy bản sao **xa nhất** (hoặc bất kỳ) của ký tự cần

In [17]:
def wrong_farthest(num, k):
    b = list(num)
    for _ in range(k): next_permutation(b)
    cur = list(num); ans = 0
    for i in range(len(cur)):
        if cur[i] == b[i]: continue
        j = len(cur) - 1
        while cur[j] != b[i]: j -= 1                     # SAI: bản sao xa nhất
        while j > i:
            cur[j], cur[j - 1] = cur[j - 1], cur[j]; j -= 1; ans += 1
    return ans

print("sai :", wrong_farthest("12121", 2), "| đúng:", Solution().getMinSwaps("12121", 2), "| BFS:", swaps_bruteforce("12121", 2))

sai : 6 | đúng: 2 | BFS: 2


**Vì sao sai:** kéo bản sao xa phải vượt qua bản sao gần (cùng ký tự) — lãng phí. **Sửa:** bản sao **gần nhất** bên phải (tương đương ghép theo thứ tự xuất hiện).

### Lỗi 2 — Đếm số vị trí khác nhau (Hamming) thay vì số lần đổi kề

In [18]:
def wrong_hamming(num, k):
    b = list(num)
    for _ in range(k): next_permutation(b)
    return sum(x != y for x, y in zip(num, b))

for num, k, exp in examples:
    print("sai :", wrong_hamming(num, k), "| đúng:", exp)

sai : 3 | đúng: 2
sai : 2 | đúng: 4
sai : 2 | đúng: 1


**Vì sao sai:** một ký tự phải đi qua nhiều vị trí, mỗi bước là một lần đổi; và một lần đổi sửa được 2 vị trí. **Sửa:** đếm nghịch thế / mô phỏng.

### Lỗi 3 — `next_permutation` dùng `>` thay vì `>=` (sai khi có chữ số lặp)

In [19]:
def wrong_next_perm(a):
    i = len(a) - 2
    while i >= 0 and a[i] > a[i + 1]:                    # SAI: phải là >=
        i -= 1
    if i < 0: return False
    j = len(a) - 1
    while a[j] <= a[i]: j -= 1
    a[i], a[j] = a[j], a[i]; a[i + 1:] = reversed(a[i + 1:]); return True

a1, a2 = list("1221"), list("1221")
next_permutation(a1)
try:
    wrong_next_perm(a2); res2 = "".join(a2)
except IndexError as e:
    res2 = f"IndexError: {e}"
print("đúng:", "".join(a1), "| sai:", res2)

đúng: 2112 | sai: IndexError: list index out of range


**Vì sao sai:** với `"1221"`, vị trí `i` cần tìm là chỗ `a[i] < a[i+1]` **nghiêm ngặt**; dùng `>` dừng ở cặp bằng nhau `2, 2` ⇒ không tìm được `j` hợp lệ (vượt mảng) hoặc ra hoán vị sai. **Sửa:** `a[i] >= a[i+1]` khi lùi.

### Lỗi 4 — Gọi `next_permutation` `k − 1` lần (lệch một)

In [20]:
def wrong_k_minus_1(num, k):
    b = list(num)
    for _ in range(k - 1): next_permutation(b)           # SAI: wonderful thứ 1 là next_permutation 1 lần
    return inversions_bit(match_positions(num, b))

for num, k, exp in examples:
    print("sai :", wrong_k_minus_1(num, k), "| đúng:", exp)

sai : 1 | đúng: 2
sai : 3 | đúng: 4
sai : 0 | đúng: 1


**Vì sao sai:** số wonderful **thứ 1** đã lớn hơn `num` ⇒ cần đúng `k` lần. **Sửa:** `range(k)`.

### Lỗi 5 — Dùng `itertools.permutations` cho `n = 1000`

Sinh hoán vị là `n!` — không bao giờ xong. Chỉ dùng để **kiểm chứng** với `n ≤ 8`. Đo thử với `n = 9` để thấy tốc độ tăng:

In [21]:
for n in (7, 8, 9):
    s = "123456789"[:n]
    t = time.perf_counter(); c = sum(1 for _ in permutations(s)); print(f"n={n}: {c} hoán vị, {(time.perf_counter()-t)*1000:.0f} ms")

n=7: 5040 hoán vị, 0 ms
n=8: 40320 hoán vị, 2 ms
n=9: 362880 hoán vị, 18 ms


**Sửa:** `next_permutation` `k` lần (`O(k·n)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số wonderful thứ k / hoán vị lớn hơn thứ k" ⇒ **next permutation k lần** (khi `k` nhỏ).
- "Số lần đổi **kề** tối thiểu để biến A thành B" ⇒ **số nghịch thế** của phép ghép vị trí; ký tự lặp ghép theo thứ tự xuất hiện.
- `n ≤ 1000` ⇒ `O(n²)` nổi bọt là đủ; `n ≤ 10⁵` ⇒ Fenwick / merge sort.
- Cẩn thận `>=` trong next permutation khi có phần tử lặp.

## 15. Biến thể của bài

1. **31 Next Permutation**: chỉ phần 1.
2. **Minimum Adjacent Swaps to Make a String Palindrome (2193)**: ghép tham lam hai đầu + đếm đổi kề.
3. **1703 Minimum Adjacent Swaps for K Consecutive Ones**: đổi kề, tổng khoảng cách tới trung vị.
4. **Đổi bất kỳ hai vị trí** (không cần kề): số lần đổi tối thiểu = `n − số chu trình` của hoán vị (khi phần tử phân biệt) — Bài tập 5.
5. **Số wonderful thứ k với k lớn (10¹⁸)**: dùng hệ cơ số giai thừa (factorial number system) trên multiset.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Liệt kê + BFS | Định nghĩa thuần | `O(n!·n)` | `O(n!)` | Kiểm chứng |
| 1b. Nổi bọt | Kéo bản gần nhất | `O(kn + n²)` | `O(n)` | Phổ biến, đủ với `n ≤ 1000` |
| 2. Ghép + Fenwick | Nghịch thế | `O(kn + n log n)` | `O(n)` | **Bản nộp** |
| 3. Ghép + merge sort | Nghịch thế | `O(kn + n log n)` | `O(n)` | Không cần BIT |
| 4. Ghép + đếm cặp | Nghịch thế `O(n²)` | `O(kn + n²)` | `O(n)` | Giải thích trực quan |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("12", 1)`; (b) `("132", 1)`; (c) `("1234", 2)`; (d) `("0011", 3)`.

**Bài 2 (tính tay):** Liệt kê 3 số wonderful nhỏ nhất của `"1243"` bằng next permutation.

**Bài 3 (điền TODO):** Hoàn thiện `count_inversions_todo(p)` bằng hai vòng lặp.

**Bài 4 (tự viết lại):** Viết `min_adjacent_swaps(A, B)` cho hai chuỗi bất kỳ cùng multiset (không cần wonderful), dùng ghép + Fenwick; đối chiếu BFS.

**Bài 5 (mini challenge):** Viết `min_any_swaps(A, B)` — số lần đổi **hai vị trí bất kỳ** tối thiểu, với **các ký tự đôi một khác nhau** (đếm chu trình). Đối chiếu BFS trên chuỗi ngắn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_inversions_todo(p):
    inv = 0
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            # TODO: nếu p[i] > p[j] thì inv += 1
            pass
    return inv

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_adjacent_swaps(A, B):
    # TODO: p = match_positions(A, B); trả inversions_bit(p)
    pass

def min_any_swaps(A, B):
    # TODO: p[i] = vị trí của B[i] trong A (ký tự phân biệt); đáp án = n - số chu trình của p
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm đích. Mức 2: đếm nghịch thế. Mức 3: (a) `"12" → "21"`: `1`; (b) `"132" → "213"`: `p = [2, 0, 1]` ⇒ `2`; (c) `"1234" → "1243" → "1324"`: `p = [0, 2, 1, 3]` ⇒ `1`; (d) `"0011" → "0101" → "0110" → "1001"`: `p = [2, 0, 1, 3]` ⇒ `2`.

**Bài 2.** Mức 1: `i` lớn nhất với `a[i] < a[i+1]`. Mức 2: `"1243" → "1324"`. Mức 3: tiếp `"1342"`, `"1423"`.

**Bài 3.** Mức 1: hai vòng. Mức 2: so `p[i] > p[j]`. Mức 3: so với `inversions_bit`.

**Bài 4.** Mức 1: kiểm tra cùng multiset. Mức 2: `match_positions`. Mức 3: BFS đối chiếu với `n ≤ 7`.

**Bài 5.** Mức 1: `p[i] = A.index(B[i])`. Mức 2: duyệt chu trình bằng mảng `seen`. Mức 3: đáp án = `n − #chu trình`; BFS với cạnh "đổi hai vị trí bất kỳ".

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
b1 = [Solution().getMinSwaps(n, k) for n, k in [("12", 1), ("132", 1), ("1234", 2), ("0011", 3)]]
print("Bài 1:", b1); assert b1 == [1, 2, 1, 2]
assert b1 == [swaps_bruteforce(n, k) for n, k in [("12", 1), ("132", 1), ("1234", 2), ("0011", 3)]]
a = list("1243"); seq = []
for _ in range(3): next_permutation(a); seq.append("".join(a))
print("Bài 2:", seq); assert seq == ["1324", "1342", "1423"]
def count_inversions_todo(p):
    inv = 0
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            if p[i] > p[j]: inv += 1
    return inv
rng = random.Random(3)
for _ in range(200):
    p = list(range(rng.randint(1, 9))); rng.shuffle(p)
    assert count_inversions_todo(p) == inversions_bit(p) == inversions_merge(p)
print("Bài 3: OK")

Bài 1: [1, 2, 1, 2]
Bài 2: ['1324', '1342', '1423']
Bài 3: OK


In [25]:
# Bài 4, 5
def min_adjacent_swaps(A, B):
    assert sorted(A) == sorted(B)
    return inversions_bit(match_positions(A, B))

def min_any_swaps(A, B):
    p = [A.index(c) for c in B]; n = len(p); seen = [False] * n; cycles = 0
    for i in range(n):
        if not seen[i]:
            cycles += 1; j = i
            while not seen[j]: seen[j] = True; j = p[j]
    return n - cycles

def any_swaps_bfs(A, B):
    dist = {A: 0}; q = deque([A])
    while q:
        s = q.popleft()
        if s == B: return dist[s]
        for i in range(len(s)):
            for j in range(i + 1, len(s)):
                t = list(s); t[i], t[j] = t[j], t[i]; t = "".join(t)
                if t not in dist: dist[t] = dist[s] + 1; q.append(t)

rng = random.Random(4)
for _ in range(200):
    A = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6))); B = "".join(rng.sample(A, len(A)))
    assert min_adjacent_swaps(A, B) == min_swaps_bfs(A, B)
    letters = rng.sample("abcdef", rng.randint(1, 6)); A2 = "".join(letters); B2 = "".join(rng.sample(letters, len(letters)))
    assert min_any_swaps(A2, B2) == any_swaps_bfs(A2, B2)
print("Bài 4, 5: OK |", min_adjacent_swaps("aabc", "cbaa"), min_any_swaps("abcd", "badc"))

Bài 4, 5: OK | 5 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
b = list(num)
for _ in range(k): next_permutation(b)           # dùng >= khi lùi tìm i
pos = defaultdict(deque)
for i, c in enumerate(num): pos[c].append(i)
p = [pos[c].popleft() for c in b]                # ghép theo thứ tự xuất hiện
return inversions(p)                             # Fenwick / merge sort / O(n^2)
```

**Checklist:**
- [ ] Gọi `next_permutation` đúng **k** lần?
- [ ] `next_permutation` dùng `a[i] >= a[i+1]` (xử lý chữ số lặp)?
- [ ] Ghép ký tự lặp theo **thứ tự xuất hiện** (bản gần nhất)?
- [ ] Đếm **số lần đổi kề** (nghịch thế), không phải số vị trí khác nhau?
- [ ] Đã test chữ số lặp nhiều (`"11112"`), số 0 ở đầu (`"00123"`), `n = 1000`, `k = 1000`?